# Sanity Check: Regulatory Ingestion (CELLAR)

Load latest Parquet from `data/regulatory/raw` and inspect basic stats.

**Note:** This notebook loads the **most recent file by modification time**, not alphabetically. If you see empty DataFrames, it means:
1. No extraction has been run yet, OR
2. The latest extraction returned 0 results (too restrictive filters)

**To run a new extraction:**
```bash
python -m src.forecasting.regulatory.ingest.run_ingest \
    query_terms=[] \
    date_from=2024-01-01 \
    date_to=2024-12-31 \
    max_pages=10
```

In [2]:
import pandas as pd
from pathlib import Path
import os

# Get project root directory (one level up from notebooks/)
notebook_dir = Path.cwd()
if notebook_dir.name == 'notebooks':
    project_root = notebook_dir.parent
else:
    project_root = notebook_dir

# Path to raw data directory
raw_dir = project_root / 'data' / 'regulatory' / 'raw'

print(f'Looking in: {raw_dir}')
print(f'Directory exists: {raw_dir.exists()}')

if raw_dir.exists():
    files = sorted(raw_dir.glob('*.parquet'), key=lambda f: f.stat().st_mtime, reverse=True)
    
    if files:
        latest_file = files[0]
        print(f'Loading: {latest_file.name}')
        df = pd.read_parquet(latest_file)
        print(f'Rows: {len(df)}')
    else:
        print('No parquet files found in directory')
        df = pd.DataFrame()
else:
    print(f'Directory not found: {raw_dir}')
    print('Run extraction first:')
    print('  python -m src.forecasting.regulatory.ingest.run_ingest query_terms=[] max_pages=5')
    df = pd.DataFrame()

df.head()

Looking in: c:\Users\fraca\forecast_refactored\data\regulatory\raw
Directory exists: True
Loading: cellar_20260105_094309.parquet
Rows: 143


,work_uri,celex,title,abstract,date_document,resource_type,eurovoc,link_html,link_pdf,language
0,http://publications.europa.eu/resource/cellar/...,72024L1416POL_202406132,None,None,2024-12-27,L,,https://eur-lex.europa.eu/legal-content/EN/TXT...,https://eur-lex.europa.eu/legal-content/EN/TXT...,None
1,http://publications.europa.eu/resource/cellar/...,72022L2555EST_202503737,None,None,2024-12-30,L,,https://eur-lex.europa.eu/legal-content/EN/TXT...,https://eur-lex.europa.eu/legal-content/EN/TXT...,None
2,http://publications.europa.eu/resource/cellar/...,72018L0851EST_202501991,None,None,2024-12-31,L,,https://eur-lex.europa.eu/legal-content/EN/TXT...,https://eur-lex.europa.eu/legal-content/EN/TXT...,None
3,http://publications.europa.eu/resource/cellar/...,72020L1828EST_202500062,None,None,2024-12-31,L,,https://eur-lex.europa.eu/legal-content/EN/TXT...,https://eur-lex.europa.eu/legal-content/EN/TXT...,None
4,http://publications.europa.eu/resource/cellar/...,72023R1113ITA_202406167,None,None,2024-12-28,L,,https://eur-lex.europa.eu/legal-content/EN/TXT...,https://eur-lex.europa.eu/legal-content/EN/TXT...,None


In [ ]:
if not df.empty:
    print('=' * 80)
    print('SUMMARY STATISTICS')
    print('=' * 80)
    print(f'Total documents: {len(df):,}')
    print(f'Date range: {df["date_document"].min()} to {df["date_document"].max()}')
    print(f'\nResource types:')
    print(df['resource_type'].value_counts())
    print(f'\nLanguages (top 10):')
    print(df['language'].value_counts().head(10))
    print(f'\nCELEX coverage: {df["celex"].notna().sum():,} ({df["celex"].notna().mean()*100:.1f}%)')
else:
    print('DataFrame is empty')

In [ ]:
if not df.empty:
    # Display columns: celex, date, type, and links
    display_cols = ['celex', 'date_document', 'resource_type', 'link_html']
    print('\nSample documents (first 10):')
    display(df[display_cols].head(10))
else:
    print('No data to display')

In [ ]:
# Document timeline visualization
if not df.empty and len(df) > 1:
    import matplotlib.pyplot as plt
    
    # Convert date column to datetime
    df['date_dt'] = pd.to_datetime(df['date_document'])
    
    # Group by date
    timeline = df.groupby('date_dt').size()
    
    plt.figure(figsize=(12, 4))
    plt.plot(timeline.index, timeline.values, marker='o', linewidth=2)
    plt.title('Documents by Publication Date')
    plt.xlabel('Date')
    plt.ylabel('Number of Documents')
    plt.grid(True, alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
    
    print(f'\nMost active dates:')
    print(timeline.nlargest(5))